# Multi-Stock Forecasting with LightGBM

This notebook demonstrates stock price forecasting for multiple stocks using LightGBM with comprehensive feature engineering.

## Features
- **Multi-stock support**: Forecast multiple stocks simultaneously
- **Weekly data aggregation**: Convert daily data to weekly frequency
- **Comprehensive feature engineering**: Lags, rolling statistics, technical indicators, Fourier transforms
- **LightGBM forecasting**: Advanced gradient boosting for time series
- **4-week ahead predictions**: Multi-step forecasting
- **Performance visualization**: Compare forecasts across stocks


## 1. Setup and Configuration


In [ ]:
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Add project root to path
project_root = os.path.abspath('..')
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Import our modules
from src.data_preprocess.stock_data_loader import StockDataLoader
from src.forecasting import WeeklyAggregator, DynamicFeatureEngineer, LightGBMForecaster

# Set up plotting
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("✅ Setup complete!")


In [ ]:
# Configuration
STOCK_SYMBOLS = ['AAPL', 'MSFT', 'GOOGL', 'TSLA', 'NVDA']  # Multiple stocks to forecast
FORECAST_HORIZON = 4  # 4 weeks ahead

print(f"🎯 Target Stocks: {', '.join(STOCK_SYMBOLS)}")
print(f"📈 Forecast Horizon: {FORECAST_HORIZON} weeks")
print(f"📊 Total Stocks: {len(STOCK_SYMBOLS)}")


## 2. Data Loading and Preparation


In [ ]:
# Load data for multiple stocks
print(f"📥 Loading data for {len(STOCK_SYMBOLS)} stocks...")
data_loader = StockDataLoader()

try:
    # Load all stocks at once
    stock_data = data_loader.load_saved_data(STOCK_SYMBOLS)
    
    print(f"✅ Successfully loaded data for {len(stock_data)} stocks")
    
    # Display summary for each stock
    print(f"\n📊 Data Summary by Stock:")
    for symbol in STOCK_SYMBOLS:
        if symbol in stock_data:
            data = stock_data[symbol]
            print(f"   • {symbol}: {len(data)} days, "
                  f"${data['Close'].mean():.2f} avg price, "
                  f"{data['Volume'].mean():,.0f} avg volume")
        else:
            print(f"   • {symbol}: ❌ No data found")
    
    # Check date ranges
    print(f"\n📅 Date Ranges:")
    for symbol in STOCK_SYMBOLS:
        if symbol in stock_data:
            data = stock_data[symbol]
            if 'Date' in data.columns:
                data['Date'] = pd.to_datetime(data['Date'])
                print(f"   • {symbol}: {data['Date'].min().strftime('%Y-%m-%d')} to {data['Date'].max().strftime('%Y-%m-%d')}")
    
except Exception as e:
    print(f"❌ Error loading data: {e}")
    raise


## 3. Weekly Data Aggregation for All Stocks


In [ ]:
# Convert daily data to weekly frequency for all stocks
print("📊 Converting all stocks to weekly data...")

weekly_aggregator = WeeklyAggregator(
    price_columns=['Open', 'High', 'Low', 'Close'],
    volume_columns=['Volume']
)

weekly_data = {}
for symbol in STOCK_SYMBOLS:
    if symbol in stock_data:
        print(f"   Processing {symbol}...")
        daily_data = stock_data[symbol].copy()
        
        # Convert Date column to datetime and set as index
        if 'Date' in daily_data.columns:
            daily_data['Date'] = pd.to_datetime(daily_data['Date'])
            daily_data.set_index('Date', inplace=True)
        
        # Aggregate to weekly
        weekly_data[symbol] = weekly_aggregator.aggregate(daily_data)
        print(f"   ✅ {symbol}: {len(weekly_data[symbol])} weeks")

print(f"\n✅ Weekly aggregation complete for {len(weekly_data)} stocks")

# Display summary
print(f"\n📋 Weekly Data Summary:")
for symbol, data in weekly_data.items():
    print(f"   • {symbol}: {data.index.min().strftime('%Y-%m-%d')} to {data.index.max().strftime('%Y-%m-%d')}, "
          f"${data['Close'].mean():.2f} avg weekly close")


## 4. Feature Engineering for All Stocks


In [ ]:
# Configure feature engineering
feature_config = {
    'fourier': {
        'n_components': 5, 
        'columns': ['Close']
    },
    'lags': {
        'lags': [1, 2, 4, 8], 
        'columns': ['Close']
    },
    'rolling': {
        'windows': [4, 8, 12], 
        'columns': ['Close'], 
        'statistics': ['mean', 'std', 'min', 'max']
    },
    'technical': {
        'indicators': ['sma', 'ema', 'rsi', 'macd', 'bollinger']
    },
    'time': {
        'features': ['day_of_week', 'month', 'quarter', 'year']
    },
    'difference': {
        'differences': [1, 2, 4],
        'columns': ['Close']
    }
}

print("🔧 Feature Engineering Configuration:")
for feature_type, config in feature_config.items():
    print(f"   • {feature_type}: {config}")


In [ ]:
# Create feature engineering pipeline for all stocks
print("\n🔧 Creating feature engineering pipeline for all stocks...")

feature_engineer = DynamicFeatureEngineer(
    forecast_horizon=FORECAST_HORIZON,
    target_column='Close',
    feature_engineering_config=feature_config
)

# Process each stock
forecasting_data = {}
for symbol in STOCK_SYMBOLS:
    if symbol in weekly_data:
        print(f"   Processing {symbol}...")
        try:
            # Create forecasting dataset
            stock_forecasting_data = feature_engineer.create_forecasting_dataset(weekly_data[symbol])
            forecasting_data[symbol] = stock_forecasting_data
            print(f"   ✅ {symbol}: {len(stock_forecasting_data)} rows, {len(stock_forecasting_data.columns)} features")
        except Exception as e:
            print(f"   ❌ {symbol}: Error in feature engineering - {e}")

print(f"\n✅ Feature engineering complete for {len(forecasting_data)} stocks")

# Display feature information
if forecasting_data:
    sample_stock = list(forecasting_data.keys())[0]
    sample_data = forecasting_data[sample_stock]
    feature_columns = [col for col in sample_data.columns 
                      if not col.startswith('target_') and col != 'Close']
    
    print(f"\n📊 Generated Features (sample from {sample_stock}):")
    print(f"   • Total features: {len(feature_columns)}")
    print(f"   • Feature types: {set([col.split('_')[0] for col in feature_columns])}")
    print(f"   • Sample features: {feature_columns[:10]}")


## 5. Model Training and Forecasting for All Stocks


In [ ]:
# Train models and make predictions for all stocks
print("🤖 Training models and making predictions for all stocks...")

models = {}
predictions = {}
training_summaries = {}

for symbol in STOCK_SYMBOLS:
    if symbol in forecasting_data:
        print(f"\n📈 Processing {symbol}...")
        
        try:
            # Prepare training data
            training_data = forecasting_data[symbol].dropna()
            
            if len(training_data) < 20:  # Need minimum data for training
                print(f"   ⚠️ {symbol}: Insufficient data for training ({len(training_data)} rows)")
                continue
            
            # Prepare features and targets
            feature_columns = [col for col in training_data.columns 
                              if not col.startswith('target_') and col != 'Close'
                              and pd.api.types.is_numeric_dtype(training_data[col])]
            
            X = training_data[feature_columns]
            y = training_data[f'target_{FORECAST_HORIZON}w']
            
            print(f"   📊 Training data: {X.shape[0]} samples, {X.shape[1]} features")
            
            # Create and train LightGBM model
            lgbm_forecaster = LightGBMForecaster(
                forecast_horizon=FORECAST_HORIZON,
                target_column='Close',
                hyperparameter_tuning=False,  # Use default parameters for faster execution
                verbose=False
            )
            
            # Train the model
            lgbm_forecaster.fit(X, y)
            models[symbol] = lgbm_forecaster
            
            # Make prediction using the last available data
            latest_data = forecasting_data[symbol].iloc[-1:]
            X_pred = latest_data[feature_columns]
            pred = lgbm_forecaster.predict(X_pred)
            predictions[symbol] = pred[0]
            
            # Store training summary
            current_price = latest_data['Close'].iloc[0]
            price_change = pred[0] - current_price
            price_change_pct = (price_change / current_price) * 100
            
            training_summaries[symbol] = {
                'current_price': current_price,
                'predicted_price': pred[0],
                'price_change': price_change,
                'price_change_pct': price_change_pct,
                'training_samples': len(training_data),
                'features_used': len(feature_columns)
            }
            
            print(f"   ✅ {symbol}: ${current_price:.2f} → ${pred[0]:.2f} ({price_change_pct:+.2f}%)")
            
        except Exception as e:
            print(f"   ❌ {symbol}: Error in training/prediction - {e}")

print(f"\n✅ Model training and prediction complete for {len(models)} stocks")


## 6. Results Summary and Comparison


In [ ]:
# Create comprehensive results summary
print("=" * 80)
print("📊 MULTI-STOCK FORECASTING RESULTS")
print("=" * 80)

print(f"\n🎯 Analysis Summary:")
print(f"   • Stocks Analyzed: {len(STOCK_SYMBOLS)}")
print(f"   • Successfully Forecasted: {len(predictions)}")
print(f"   • Forecast Horizon: {FORECAST_HORIZON} weeks ahead")
print(f"   • Analysis Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

print(f"\n💰 Price Forecasts:")
print(f"{'Stock':<8} {'Current':<10} {'Predicted':<10} {'Change':<10} {'Change %':<10}")
print("-" * 60)

for symbol in STOCK_SYMBOLS:
    if symbol in training_summaries:
        summary = training_summaries[symbol]
        print(f"{symbol:<8} ${summary['current_price']:<9.2f} ${summary['predicted_price']:<9.2f} "
              f"${summary['price_change']:<9.2f} {summary['price_change_pct']:<9.2f}%")
    else:
        print(f"{symbol:<8} {'N/A':<10} {'N/A':<10} {'N/A':<10} {'N/A':<10}")

# Calculate portfolio-level statistics
if predictions:
    current_prices = [training_summaries[s]['current_price'] for s in training_summaries]
    predicted_prices = [training_summaries[s]['predicted_price'] for s in training_summaries]
    price_changes = [training_summaries[s]['price_change'] for s in training_summaries]
    price_changes_pct = [training_summaries[s]['price_change_pct'] for s in training_summaries]
    
    print(f"\n📈 Portfolio Statistics:")
    print(f"   • Average Current Price: ${np.mean(current_prices):.2f}")
    print(f"   • Average Predicted Price: ${np.mean(predicted_prices):.2f}")
    print(f"   • Average Price Change: ${np.mean(price_changes):.2f}")
    print(f"   • Average Change %: {np.mean(price_changes_pct):+.2f}%")
    print(f"   • Bullish Stocks: {sum(1 for pct in price_changes_pct if pct > 0)}")
    print(f"   • Bearish Stocks: {sum(1 for pct in price_changes_pct if pct < 0)}")

print(f"\n🔧 Model Information:")
for symbol in STOCK_SYMBOLS:
    if symbol in training_summaries:
        summary = training_summaries[symbol]
        print(f"   • {symbol}: {summary['training_samples']} training samples, {summary['features_used']} features")

print("\n" + "=" * 80)
print("✅ Multi-stock forecasting completed successfully!")
print("=" * 80)


## 7. Visualization


In [ ]:
# Create comprehensive visualization
fig, axes = plt.subplots(2, 2, figsize=(20, 16))
fig.suptitle('Multi-Stock Forecasting Analysis', fontsize=20, fontweight='bold')

# 1. Price history and forecasts for all stocks
ax1 = axes[0, 0]
colors = plt.cm.Set1(np.linspace(0, 1, len(STOCK_SYMBOLS)))

for i, symbol in enumerate(STOCK_SYMBOLS):
    if symbol in weekly_data:
        data = weekly_data[symbol]
        ax1.plot(data.index, data['Close'], label=f'{symbol} Historical', 
                color=colors[i], alpha=0.7, linewidth=1.5)
        
        # Add forecast point
        if symbol in predictions:
            latest_date = data.index[-1]
            ax1.scatter(latest_date, predictions[symbol], 
                       color=colors[i], s=100, marker='*', 
                       label=f'{symbol} Forecast', zorder=5)

ax1.set_title('Price History and 4-Week Forecasts', fontsize=14, fontweight='bold')
ax1.set_ylabel('Price ($)')
ax1.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
ax1.grid(True, alpha=0.3)

# 2. Forecast percentage changes
ax2 = axes[0, 1]
if predictions:
    symbols_with_forecasts = [s for s in STOCK_SYMBOLS if s in training_summaries]
    changes_pct = [training_summaries[s]['price_change_pct'] for s in symbols_with_forecasts]
    colors_changes = ['green' if x > 0 else 'red' for x in changes_pct]
    
    bars = ax2.bar(symbols_with_forecasts, changes_pct, color=colors_changes, alpha=0.7)
    ax2.set_title('4-Week Forecast: Percentage Change', fontsize=14, fontweight='bold')
    ax2.set_ylabel('Change (%)')
    ax2.axhline(y=0, color='black', linestyle='-', alpha=0.3)
    ax2.grid(True, alpha=0.3)
    
    # Add value labels on bars
    for bar, value in zip(bars, changes_pct):
        height = bar.get_height()
        ax2.text(bar.get_x() + bar.get_width()/2., height + (0.1 if height > 0 else -0.3),
                f'{value:+.1f}%', ha='center', va='bottom' if height > 0 else 'top')

# 3. Current vs Predicted prices
ax3 = axes[1, 0]
if predictions:
    current_prices = [training_summaries[s]['current_price'] for s in symbols_with_forecasts]
    predicted_prices = [training_summaries[s]['predicted_price'] for s in symbols_with_forecasts]
    
    x = np.arange(len(symbols_with_forecasts))
    width = 0.35
    
    bars1 = ax3.bar(x - width/2, current_prices, width, label='Current Price', alpha=0.7, color='blue')
    bars2 = ax3.bar(x + width/2, predicted_prices, width, label='Predicted Price', alpha=0.7, color='orange')
    
    ax3.set_title('Current vs Predicted Prices', fontsize=14, fontweight='bold')
    ax3.set_ylabel('Price ($)')
    ax3.set_xticks(x)
    ax3.set_xticklabels(symbols_with_forecasts)
    ax3.legend()
    ax3.grid(True, alpha=0.3)

# 4. Training data quality
ax4 = axes[1, 1]
if predictions:
    training_samples = [training_summaries[s]['training_samples'] for s in symbols_with_forecasts]
    features_used = [training_summaries[s]['features_used'] for s in symbols_with_forecasts]
    
    ax4_twin = ax4.twinx()
    
    bars1 = ax4.bar(x - width/2, training_samples, width, label='Training Samples', alpha=0.7, color='green')
    bars2 = ax4_twin.bar(x + width/2, features_used, width, label='Features Used', alpha=0.7, color='purple')
    
    ax4.set_title('Model Training Quality', fontsize=14, fontweight='bold')
    ax4.set_ylabel('Training Samples', color='green')
    ax4_twin.set_ylabel('Features Used', color='purple')
    ax4.set_xticks(x)
    ax4.set_xticklabels(symbols_with_forecasts)
    
    # Add legends
    lines1, labels1 = ax4.get_legend_handles_labels()
    lines2, labels2 = ax4_twin.get_legend_handles_labels()
    ax4.legend(lines1 + lines2, labels1 + labels2, loc='upper left')
    
    ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## 8. Next Steps and Recommendations

**Portfolio Analysis:**
- **Top Performers**: Focus on stocks with highest predicted returns
- **Risk Management**: Consider volatility and correlation between stocks
- **Diversification**: Balance between different sectors and market caps

**Model Improvements:**
1. **Hyperparameter Tuning**: Use `StandaloneBacktester` for model validation
2. **Ensemble Methods**: Combine multiple models for better predictions
3. **Market Regime Detection**: Adapt models based on market conditions
4. **Feature Selection**: Optimize feature sets for each stock individually

**Production Deployment:**
1. **Automated Updates**: Set up daily data refresh and model retraining
2. **Performance Monitoring**: Track prediction accuracy over time
3. **Alert System**: Notify on significant price movements or model changes
4. **Portfolio Integration**: Use forecasts in broader investment strategies

**Risk Considerations:**
- ⚠️ **Past Performance**: Historical data doesn't guarantee future results
- ⚠️ **Market Volatility**: External factors can significantly impact predictions
- ⚠️ **Model Limitations**: Machine learning models have inherent uncertainties
- ⚠️ **Diversification**: Don't rely on single stock predictions for investment decisions
